# Load model

For now, we are using a pretrained [PredRNN-V2](https://github.com/thuml/predrnn-pytorch) model.

## Set up env

In [ ]:
# prompt: add a choice param for which dataset it was pretrained on mnist or kth or bair

pretraining_dataset = "mnist" #@param ["mnist", "kth", "bair"]

# # Load model
#
# For now, we are using a pretrained [PredRNN-V2](https://github.com/thuml/predrnn-pytorch) model.
# Now using the selected dataset: {dataset}

Clone repo

In [ ]:
!git clone https://github.com/thuml/predrnn-pytorch.git
%cd predrnn-pytorch

Cloning into 'predrnn-pytorch'...
remote: Enumerating objects: 474, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 474 (delta 105), reused 86 (delta 86), pack-reused 350 (from 1)
Receiving objects: 100% (474/474), 27.57 MiB | 22.00 MiB/s, done.
Resolving deltas: 100% (233/233), done.
/content/predrnn-pytorch


Installs

In [ ]:
!pip install -q gdown

Download pretrained model checkpoints

In [ ]:
!mkdir -p checkpoints

# MNIST model
!gdown 1xZT_KYZDGg7JCXotp-rG6KaANNmybfiX -O checkpoints/mnist_model.ckpt

# KTH model
!gdown 1h0ngXMbOxtEJxRVG0Isj6ecfaIqhRdKr -O checkpoints/kth_model.ckpt

# BAIR model
!gdown 1g5WvaAtW9rjYhP94ohZXFyqLgj7jpRzg -O checkpoints/bair_model.ckpt

Downloading...
From: https://drive.google.com/uc?id=1xZT_KYZDGg7JCXotp-rG6KaANNmybfiX
To: /content/predrnn-pytorch/checkpoints/mnist_model.ckpt
100% 95.4M/95.4M [00:00<00:00, 197MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1h0ngXMbOxtEJxRVG0Isj6ecfaIqhRdKr
From (redirected): https://drive.google.com/uc?id=1h0ngXMbOxtEJxRVG0Isj6ecfaIqhRdKr&confirm=t&uuid=8f90e1c2-472a-44cb-9492-4a245e1f394e
To: /content/predrnn-pytorch/checkpoints/kth_model.ckpt
100% 95.4M/95.4M [00:00<00:00, 192MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1g5WvaAtW9rjYhP94ohZXFyqLgj7jpRzg
From (redirected): https://drive.google.com/uc?id=1g5WvaAtW9rjYhP94ohZXFyqLgj7jpRzg&confirm=t&uuid=da24862a-f148-4a0c-ba82-4ecd567a0e2e
To: /content/predrnn-pytorch/checkpoints/bair_model.ckpt
100% 134M/134M [00:00<00:00, 243MB/s]


Imports

In [ ]:
import numpy as np
import torch
from core.models.model_factory import Model
from core.utils.preprocess import reshape_patch, reshape_patch_back

## Load pretrained model

In [ ]:
class Configs:
    def __init__(self):
        # Defaults
        self.model_name = 'predrnn_v2'
        self.num_hidden = '128,128,128,128'
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.input_length = 10
        self.total_length = 20
        self.patch_size = 4
        self.layer_norm = False
        self.reverse_input = True
        self.model_path = None
        self.batch_size = 4
        self.save_dir = 'checkpoints'
        self.lr = 0.0001

        # Required by model
        self.visual = False
        self.visual_path = './visual'
        self.filter_size = 5
        self.stride = 1
        self.reverse_scheduled_sampling = True
        self.r_sampling_step_1 = 5000
        self.r_sampling_step_2 = 50000
        self.r_exp_alpha = 2000

        # Dataset-specific extras
        self.conv_on_input = False
        self.res_on_conv = False
        self.decouple_beta = 0.0
        self.img_width = 64
        self.img_channel = 1

    @classmethod
    def from_dataset(cls, name: str) -> 'Configs':
        cfg = cls()
        if name == 'mnist':
            cfg.img_width = 64
            cfg.img_channel = 1
            cfg.decouple_beta = 0.1
            cfg.batch_size = 8
            cfg.r_sampling_step_1 = 25000
            cfg.r_exp_alpha = 2500
        elif name == 'kth':
            cfg.img_width = 128
            cfg.img_channel = 1
            cfg.decouple_beta = 0.01
        elif name == 'bair':
            cfg.model_name = 'action_cond_predrnn_v2'
            cfg.img_width = 64
            cfg.img_channel = 3
            cfg.input_length = 2
            cfg.total_length = 12
            cfg.patch_size = 1
            cfg.batch_size = 16
            cfg.decouple_beta = 0.1
            cfg.conv_on_input = True
            cfg.res_on_conv = True
            cfg.r_sampling_step_1 = 25000
            cfg.r_exp_alpha = 2500
            cfg.num_action_ch = 4
        else:
            raise ValueError(f"Unsupported dataset: {name}")
        return cfg

In [ ]:
# -----------------------------------
# Create model
# -----------------------------------
if pretraining_dataset == "mnist":
    configs = Configs.from_dataset('mnist')
elif pretraining_dataset == "kth":
    configs = Configs.from_dataset('kth')
elif pretraining_dataset == "bair":
    configs = Configs.from_dataset('bair')
else:
    raise ValueError(f"Unsupported dataset: {pretraining_dataset}")

model = Model(configs)

# Load checkpoint
model.load(f'checkpoints/{pretraining_dataset}_model.ckpt')

load model: checkpoints/mnist_model.ckpt


## Run dummy inference

In [ ]:
# # -----------------------------------
# # Run dummy inference
# # -----------------------------------
# -- Step 1: Create dummy [B, T, H, W, C] input
B, T, H, W, C = configs.batch_size, configs.total_length, configs.img_width, configs.img_width, configs.img_channel
ims = np.random.rand(B, T, H, W, C).astype(np.float32)

# -- Step 2: Patchify input [B, T, H, W, C] → [B, T, patch*C, H//p, W//p]
patch_ims = reshape_patch(ims, configs.patch_size)

# -- Step 3: Convert patchified input to torch tensor
frames = torch.from_numpy(patch_ims).to(configs.device)

# -- Step 4: Build real_input_flag tensor (same shape as frames)
mask = torch.ones_like(frames)

# -- Step 5: Run forward pass through the network
model.network.eval()
with torch.no_grad():
    pred_patches, loss = model.network(frames, mask)

# -- Step 6: Convert prediction back from patches
pred_np = pred_patches.cpu().numpy()  # [B, T-1, H//p, W//p, patch*C]
pred_frames = reshape_patch_back(pred_np, configs.patch_size)  # [B, T-1, H, W, C]

print("Predicted frame shape:", pred_frames.shape)  # Expect (B, T-1, H, W, C)

Predicted frame shape: (8, 19, 64, 64, 1)
